In [10]:
# TODO 1. Identifizieren welche Fragen sich für Rückfragen eignen.
# 2. Rückfragen erstellen
# 3. Schlüsselbegriff und ID der Frage ergänzen
# 4. CSV callback_questions generieren

In [11]:
# Datei einlesen & chunk für Unterteilung in kleinere Anfragen erstellen
import pandas as pd
from llm_handler import LLMHandler

path = "../interim/generated_q_and_a.csv"
df = pd.read_csv(path)
df.head()

,Unnamed: 0,question,answer,max_points,keywords,classification,original_question,original_answer
0,0,Was ist Data Science?,"Data Science ist ein interdisziplinäres Feld, ...",5,interdisziplinär; wissenschaftliche Methoden; ...,Hauptfrage,What is data science?,Data science is an interdisciplinary field tha...
1,1,Was sind die wesentlichen Schritte im Datenwis...,Die wesentlichen Schritte umfassen typischerwe...,6,Problemdefinition; Datenerhebung; Datenvorbere...,Aufzählung,What are the key steps in the data science pro...,The key steps typically include problem defini...
2,2,Was ist der Unterschied zwischen überwachtem u...,Überwachtes Lernen beinhaltet das Trainieren e...,5,überwachtes Lernen; beschriftete Daten; Eingan...,Hauptfrage,What is the difference between supervised and ...,Supervised learning involves training a model ...
3,3,Erkläre das Bias-Variance-Dilemma.,Das Bias-Variance-Dilemma ist das Gleichgewich...,5,Bias; Variance; Tradeoff; Underfitting; Overfi...,Hauptfrage,Explain the bias-variance tradeoff.,The bias-variance tradeoff is the balance betw...
4,4,Was ist Feature Engineering?,Feature engineering ist der Prozess der Auswah...,4,Feature Engineering; Auswahl; Transformation; ...,Hauptfrage,What is feature engineering?,Feature engineering is the process of selectin...


In [12]:
# Prompt schreiben


def construct_prompt(question, answer, keywords):
    return f"""Du bist Dozent für Data Science und möchtest für eine mündliche Prüfung Rückfragen vorbereiten.
    Diese Rückfragen sollen prüfen, ob ein Student, der Schlüsselbegriffe nicht genannt hat, diese tatsächlich nicht kennt oder nur in dem Moment nicht daran gedacht hat.
    Die Rückfrage soll dabei gezielt auf einen Schlüsselbegriff fokusieren, selbst wenn vielleicht mehr als ein Begriff nicht genannt wurde. 
    Gehe bei deiner Antwort die nachfolgenden Schritte der Reihe nach durch:
    1. Überlege, ob die Frage {question} mit der dazugehörigen Antwort {answer} für eine Rückfrage geeignet ist. 
    
    2. Generiere eine Rückfrage für jeden Schlüsselbegriff aus {keywords}, für den dies sinnvoll möglich erscheint. 
    Beachte dabei, dass der Begriff als solches nach Möglichkeit nicht direkt genannt wird, 
    sondern eine Umschreibung für die Rückfrage erfolgt, die den Studenten unterstützt, selbst auf den fehlenden Begriff oder die fehlende Erklärung zu kommen.
    Sprich den Studenten dabei mit Sie an.
        
    3. Gib deine Antwort im angegebenen Antwortformat zurück. Beachte dabei, dass die generierten Rückfragen in der gleichen Reihenfolge notiert werden, wie die Schlüsselbegriffe.

    <Antwortformat>
    ```json
    {{
        "useable": "ja | nein",
        "callback": "<callback1; callback2; callback3;...>",
        "keywords": "<keyword1; keyword2; keyword3;...>",
    }}```
    """

In [13]:
# LLM aufrufen und in Dataframe speichern

llm = LLMHandler()
result_list = []

for index, row in df.head(50).iterrows():
    prompt = construct_prompt(row["question"], row["answer"], row["keywords"])
    answer = llm.call_llm(prompt)
    answer["question_id"] = row[0]
    result_list.append(answer)

callback_data = pd.DataFrame(result_list)

/tmp/ipykernel_24662/401019323.py:9: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  answer["question_id"] = row[0]


In [15]:
# Daten CSV speichern

callback_data.to_csv("generated_callback.csv")

callback_data.head()

,useable,callback,keywords,question_id
0,ja,"Können Sie erklären, wie verschiedene wissensc...",interdisziplinär; wissenschaftliche Methoden; ...,0
1,ja,"Können Sie beschreiben, wie man mit der Festle...",Problemdefinition; Datenerhebung; Datenvorbere...,1
2,ja,"Können Sie erläutern, bei welchem Lernalgorith...",überwachtes Lernen; beschriftete Daten; Eingan...,2
3,ja,"Sie haben erwähnt, wie ein Modell zu einfach s...",Bias; Variance; Tradeoff; Underfitting; Overfi...,3
4,ja,"Können Sie beschreiben, wie man aus einem Date...",Auswahl; Transformation; Erstellung; Rohdaten;...,4
